In [ ]:
import requests          # schickt HTTP Anfragen ans Internet
import urllib3           # zum Ausschalten einer SSL Warnung
import time               # für Wartezeiten zwischen Anfragen
import csv                # speichert Daten im CSV Format
from datetime import datetime   # für Zeitstempel
from bs4 import BeautifulSoup   # parst HTML in eine durchsuchbare Struktur

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = 'https://steamcommunity.com/market/search/render/'
APP_ID = 730                  # 730 heisst Counter Strike 2 bei Steam
TAG = 'tag_weapon_ak47'       # filtert nur AK 47 Skins
COUNT = 100                   # wie viele Items wir pro Anfrage anfordern
DELAY = 1.5                   # Sekunden Pause zwischen Anfragen
MAX_RETRIES = 3                # wie oft wir eine fehlgeschlagene Anfrage wiederholen

headers = {'User-Agent': 'Mozilla/5.0'}   # gibt uns aus wie ein normaler Browser


def fetch_page(start):
    # baut die Anfrage an Steam mit allen nötigen Filtern
    params = {
        'appid': APP_ID,
        'category_730_Weapon[]': TAG,
        'count': COUNT,
        'start': start,
        'sort_column': 'name',   # feste Sortierung, sonst verschiebt sich die Reihenfolge zwischen Anfragen
        'sort_dir': 'asc',
    }

    attempt = 1
    while attempt <= MAX_RETRIES:
        # Mechanismus: bei jedem Fehlschlag zählt attempt hoch, die Schleife bricht ab sobald MAX_RETRIES erreicht ist
        try:
            response = requests.get(BASE_URL, headers=headers, params=params, verify=False, timeout=10)
            response.raise_for_status()   # löst selbst einen Fehler aus, wenn Steam mit einem Fehlercode antwortet
            return response.json()         # gibt die Antwort als Python Daten zurück
        except Exception as e:
            print(f"Versuch {attempt} fehlgeschlagen bei start={start}: {e}")
            attempt = attempt + 1
            time.sleep(DELAY)   # kurz warten, bevor der nächste Versuch startet

    print(f"Aufgegeben bei start={start}")
    return None   # None heisst, alle Versuche sind fehlgeschlagen


def parse_page(html):
    # BeautifulSoup baut aus rohem HTML eine Baumstruktur, in der man gezielt suchen kann
    soup = BeautifulSoup(html, 'html.parser')
    rows = soup.find_all('a', class_='market_listing_row_link')   # jede Zeile mit dieser Klasse ist ein Item

    items = []
    for row in rows:
        div = row.find('div', class_='market_listing_row')

        hash_name = div.get('data-hash-name')   # eindeutige Kennung des Items bei Steam
        url = row.get('href')                    # Link zur Verkaufsseite des Items

        qty_span = div.find('span', class_='market_listing_num_listings_qty')
        if qty_span:
            quantity = int(qty_span['data-qty'])   # Anzahl aktiver Angebote
        else:
            quantity = None

        price_span = div.find('span', class_='normal_price', attrs={'data-price': True})
        # sucht gezielt den Span der ein data-price Attribut besitzt, weil es zwei normal_price Spans pro Item gibt
        if price_span:
            price = int(price_span['data-price']) / 100   # Preis kommt in Cent an, Teilen durch 100 ergibt Dollar
        else:
            price = None

        name_span = div.find('span', class_='market_listing_item_name')
        if name_span:
            name = name_span.text
        else:
            name = None

        wear = None
        if name and '(' in name and name.endswith(')'):
            # rfind sucht von rechts, damit auch bei Namen mit mehreren Klammern die letzte genommen wird
            # der Ausschnitt beginnt einen Schritt nach der Klammer und endet einen Schritt vor dem Ende
            wear = name[name.rfind('(') + 1:-1]

        item = {
            'name': name,
            'hash_name': hash_name,
            'wear': wear,
            'url': url,
            'price_usd': price,
            'quantity': quantity,
            'scraped_at': datetime.now().isoformat(),
        }
        items.append(item)

    return items


def scrape_all():
    # holt Seite für Seite, bis alle Items gesammelt sind
    first = fetch_page(0)
    if first is None:
        print("Erste Seite konnte nicht geladen werden, breche ab.")
        return []

    total_count = first['total_count']   # wie viele Items es insgesamt gibt
    print(f"Total zu sammeln: {total_count}")

    all_items = parse_page(first['results_html'])
    start = len(all_items)   # zählt was wirklich angekommen ist, nicht was angefordert wurde

    while start < total_count:
        # Mechanismus: die Schleife läuft solange die Bedingung wahr ist, sie stoppt sofort sobald start nicht mehr kleiner ist
        time.sleep(DELAY)
        page = fetch_page(start)

        if page is None:
            print(f"Seite ab start={start} nicht ladbar, breche ab.")
            break

        page_items = parse_page(page['results_html'])

        if not page_items:
            print(f"Keine weiteren Items ab start={start}, breche ab.")
            break

        all_items.extend(page_items)
        start = start + len(page_items)   # zählt die echte Anzahl, nicht die angeforderte
        print(f"Gesammelt {len(all_items)} / {total_count}")

    return all_items   # Deduplikation passiert nicht mehr hier, sondern später in Pandas


if __name__ == '__main__':
    data = scrape_all()

    fieldnames = ['name', 'hash_name', 'wear', 'url', 'price_usd', 'quantity', 'scraped_at']
    with open('2-raw_learn_data.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)   # schreibt jedes Dictionary automatisch als eigene Zeile
        writer.writeheader()
        writer.writerows(data)
    print(f"{len(data)} Items in 2-raw_learn_data.csv gespeichert")

Total zu sammeln: 566
Gesammelt 20 / 566
Gesammelt 30 / 566
Gesammelt 40 / 566
Gesammelt 50 / 566
Gesammelt 60 / 566
Gesammelt 70 / 566
Gesammelt 80 / 566
Gesammelt 90 / 566
Gesammelt 100 / 566
Gesammelt 110 / 566
Gesammelt 120 / 566
Gesammelt 130 / 566
Gesammelt 140 / 566
Gesammelt 150 / 566
Gesammelt 160 / 566
Gesammelt 170 / 566
Gesammelt 180 / 566
Gesammelt 190 / 566
Gesammelt 200 / 566
Gesammelt 210 / 566
Gesammelt 220 / 566
Gesammelt 230 / 566
Gesammelt 240 / 566
Gesammelt 250 / 566
Gesammelt 260 / 566
Gesammelt 270 / 566
Gesammelt 280 / 566
Gesammelt 290 / 566
Gesammelt 300 / 566
Gesammelt 310 / 566
Gesammelt 320 / 566
Gesammelt 330 / 566
Gesammelt 340 / 566
Gesammelt 350 / 566
Gesammelt 360 / 566
Gesammelt 370 / 566
Gesammelt 380 / 566
Gesammelt 390 / 566
Gesammelt 400 / 566
Gesammelt 410 / 566
Gesammelt 420 / 566
Gesammelt 430 / 566
Gesammelt 440 / 566
Gesammelt 450 / 566
Gesammelt 460 / 566
Gesammelt 470 / 566
Gesammelt 480 / 566
Gesammelt 490 / 566
Gesammelt 500 / 566
Gesamm